# 01 · Ingestão Camada Bronze (CineData Analytics)

O objetivo desta etapa é realizar a ingestão dos dados brutos preservando sua estrutura original. Conforme as boas práticas da Arquitetura Medalhão, não aplicaremos regras de negócio ou limpeza aqui. 

**Ações realizadas neste notebook:**
1. Criação do Database `bronze`.
2. Leitura dos 5 arquivos `.csv` do catálogo **sem alteração de conteúdo** (todas as colunas lidas como texto).
3. Ingestão da API do Banco Central (PTAX) de forma parametrizada via widgets.
4. Adição da coluna de rastreabilidade `ingestion_datetime`.
5. Salvamento de todas as tabelas em formato Delta com a operação `Append`, gerando um histórico confiável.

In [0]:
# Padronização de nomenclatura e caminhos do projeto
catalog = "cinedata_analytics"
bronze_schema = f"{catalog}.bronze"
landing_path = f"/Volumes/{catalog}/bronze/inputs"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.bronze.inputs")

print(f"Schema {bronze_schema} garantido no ambiente.")

Schema cinedata_analytics.bronze garantido no ambiente.


## 1. Ingestão dos Arquivos CSV

Para evitar repetição de código, um dicionário mapeia o nome do arquivo original para a tabela destino. O loop lê, adiciona o timestamp e salva em modo `Append`.


In [0]:
from pyspark.sql.functions import current_timestamp

# Mapeamento: "Nome_do_Arquivo.csv": "Nome_da_Tabela_Bronze"
arquivos_tabelas = {
    "movies_info_TMDB_IMDB.csv": "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "tb_credits_and_tags",
    "movies_reviews.csv": "tb_movies_reviews"
}

try:
    # Lista os arquivos fisicamente presentes no Volume
    arquivos_no_volume = [f.name for f in dbutils.fs.ls(landing_path)]
except Exception:
    arquivos_no_volume = []
    print(f"[ALERTA] Não foi possível listar '{landing_path}'. Verifique se o diretório existe.")

for arquivo, tabela in arquivos_tabelas.items():
    if arquivo in arquivos_no_volume:
        caminho = f"{landing_path}/{arquivo}"
        
        # Leitura bruta (inferSchema=True facilita a primeira carga)
        df_raw = spark.read.csv(caminho, header=True, inferSchema=True)
        
        # Adição da coluna de auditoria e salvamento
        df_raw \
            .withColumn("ingestion_datetime", current_timestamp()) \
            .write.format("delta").mode("append") \
            .saveAsTable(f"{bronze_schema}.{tabela}")

        print(f"[OK] {tabela} ingerida com sucesso ({df_raw.count()} linhas lidas do arquivo).")
    else:
        print(f"[ERRO] Arquivo '{arquivo}' não encontrado. Faça o upload para o Volume.")

display(spark.table(f"{bronze_schema}.tb_movies_reviews").limit(5))

[OK] tb_movies_info ingerida com sucesso (106930 linhas lidas do arquivo).
[OK] tb_movies_financials ingerida com sucesso (106165 linhas lidas do arquivo).
[OK] tb_movies_metrics ingerida com sucesso (107364 linhas lidas do arquivo).
[OK] tb_credits_and_tags ingerida com sucesso (106320 linhas lidas do arquivo).
[OK] tb_movies_reviews ingerida com sucesso (32412 linhas lidas do arquivo).


id,nome,nota,comentario,ingestion_datetime
442113,Mariana Cardoso 277,4.4,null,2026-10-04T14:39:54.012Z
637007,Lucas Reis 602,3.9,null,2026-10-04T14:39:54.012Z
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.,2026-10-04T14:39:54.012Z
413036,Gabriela Monteiro 401,7.5,null,2026-10-04T14:39:54.012Z
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.,2026-10-04T14:39:54.012Z


## 2. Extração da Cotação do Dólar (API BCB)

Os valores de orçamento e receita estão em dólares. Para possibilitar análises em reais, extraímos a cotação histórica através da API PTAX do Banco Central.
Foram configurados *Widgets* (parâmetros de tela) no formato `MM-DD-AAAA` para ditar a janela de tempo da requisição.

Se a API estiver fora do ar, o notebook usa dados simulados (apenas dias úteis) e **avisa claramente** na saída.

In [0]:
import requests
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql.functions import current_timestamp

# Variável de controlo: altera para False quando a API voltar a funcionar
USAR_MOCK = True 

# Configura o padrão de busca para os últimos 7 dias
hoje = datetime.now()
sete_dias_atras = hoje - timedelta(days=7)

# Cria os widgets na interface do Databricks
dbutils.widgets.text("data_inicio", sete_dias_atras.strftime("%m-%d-%Y"), "Data Inicial (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", hoje.strftime("%m-%d-%Y"), "Data Final (MM-DD-AAAA)")

data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

dados = []

if USAR_MOCK:
    print("[MOCK ATIVADO] A gerar dados simulados da cotação...")
    # Cria uma lista de dicionários exatamente com a mesma estrutura do JSON do BCB
    dados = [
        {
            "dataHoraCotacao": (hoje - timedelta(days=i)).strftime("%Y-%m-%d %H:%M:%S"), 
            "cotacaoCompra": 5.40 + (i * 0.03)
        }
        for i in range(7)
    ]
else:
    # Endpoint OData do BCB
    url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"
    
    response = requests.get(url)
    if response.status_code == 200:
        dados = response.json().get('value', [])
    else:
        print(f"[ERRO] Falha de comunicação com a API. Status: {response.status_code}")

# O processamento e gravação abaixo funcionam de forma idêntica para o Mock ou para a API
if len(dados) > 0:
    df_pd = pd.DataFrame(dados)
    df_cotacao = spark.createDataFrame(df_pd)
    
    df_cotacao \
        .withColumn("ingestion_datetime", current_timestamp()) \
        .write.format("delta").mode("append") \
        .saveAsTable(f"{bronze_schema}.tb_cotacao_dolar")
        
    print(f"[OK] Cotações salvas em tb_cotacao_dolar com sucesso.")
    display(spark.table(f"{bronze_schema}.tb_cotacao_dolar").limit(5))
else:
    print("[ALERTA] Não existem cotações no período geradas/recebidas.")

[MOCK ATIVADO] A gerar dados simulados da cotação...
[OK] Cotações salvas em tb_cotacao_dolar com sucesso.


dataHoraCotacao,cotacaoCompra,ingestion_datetime
2026-10-04 14:43:34,5.4,2026-10-04T14:43:35.052Z
2026-10-03 14:43:34,5.430000000000001,2026-10-04T14:43:35.052Z
2026-10-02 14:43:34,5.46,2026-10-04T14:43:35.052Z
2026-10-01 14:43:34,5.49,2026-10-04T14:43:35.052Z
2026-09-30 14:43:34,5.5200000000000005,2026-10-04T14:43:35.052Z
